# Buoc 1: Load du lieu

### Load tu SSI

In [ ]:
# Import necessary modules
from ssi_fc_data import fc_md_client, model
import pandas as pd  # Import Pandas for DataFrame handling
import json
import configDataSSI

# Create a Market Data Client
from_date = "15/03/2025"
to_date = "18/03/2025"
client = fc_md_client.MarketDataClient(configDataSSI)

req = model.daily_ohlc('VCB', from_date, to_date)

data_dict = client.daily_ohlc(configDataSSI, req)

# Access the list of dictionaries in the "data" field
data_list = data_dict['data']

# Convert the list of dictionaries into a DataFrame
data = pd.DataFrame(data_list, columns=['TradingDate', 'Open', 'High', 'Low', 'Close', 'Volume'])
data = data.rename(columns={'TradingDate': 'Datetime'})

# Print or work with the DataFrame
data

### Load tu YFinance

In [1]:
from datetime import datetime, timedelta
import schedule
import time
import sys
sys.path.append('../Common')

import CommonYFinance

symbol = 'VCB.VN'
# from_date = '2024-11-01'
# to_date = '2024-10-16' # yyyy-mm-dd # to_date = to - 1

from_date = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua

timeframe = '1d'
data = CommonYFinance.CommonYFinance.loaddataYFinance(symbol, from_date, to_date, timeframe)

data

[*********************100%***********************]  1 of 1 completed


,Datetime,Adj Close,Close,High,Low,Open,Volume
0,2025-03-17,67300.0,67300.0,67300.0,66000.0,66000.0,5003800
1,2025-03-18,66800.0,66800.0,67800.0,66800.0,67800.0,3757000


In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Datetime   2 non-null      datetime64[ns]
 1   Adj Close  2 non-null      float64       
 2   Close      2 non-null      float64       
 3   High       2 non-null      float64       
 4   Low        2 non-null      float64       
 5   Open       2 non-null      float64       
 6   Volume     2 non-null      int64         
dtypes: datetime64[ns](1), float64(5), int64(1)
memory usage: 244.0 bytes


# Buoc 2: Viet ham de check chien luoc (Viet 3 ham kiem tra du lieu)

In [3]:
import pandas as pd
# data o tren

data['Datetime'] = pd.to_datetime(data['Datetime'], dayfirst=True)
data.set_index('Datetime', inplace=True)

# Giả sử 'data' là DataFrame của bạn với dữ liệu lịch sử giá cổ phiếu
data['Open'] = pd.to_numeric(data['Open'], errors='coerce')
data['High'] = pd.to_numeric(data['High'], errors='coerce')
data['Low'] = pd.to_numeric(data['Low'], errors='coerce')
data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
data['Volume'] = pd.to_numeric(data['Volume'], errors='coerce')

# Định nghĩa hàm để kiểm tra nến Doji chân dài
def is_long_legged_doji(row): # row la 1 dong cua data
    body_range = abs(row['Close'] - row['Open']) # Doji khong phan biet Open > Close hay Close > Open
    upper_shadow = row['High'] - max(row['Open'], row['Close']) # Bong nen tren
    lower_shadow = min(row['Open'], row['Close']) - row['Low'] # Bong nen duoi
    
    # Điều chỉnh ngưỡng này theo dữ liệu cụ thể của bạn
    doji_threshold = 5 / 100 * row['Close'] # 5% của giá đóng cửa
    return body_range <= doji_threshold and upper_shadow >= 2 * body_range and lower_shadow >= 2 * body_range

# Định nghĩa hàm để kiểm tra nến tăng
def is_bullish_candle(current_row, previous_row):
    return (current_row['Close'] > current_row['Open'] and # Close > Open la nến tăng
            current_row['Close'] > previous_row['Close'] and
            previous_row['Close'] <= previous_row['Open'])

# Định nghĩa hàm để kiểm tra nến giảm
def is_bearish_candle(current_row, previous_row):
    return (current_row['Close'] < current_row['Open'] and
            current_row['Close'] < previous_row['Close'] and
            previous_row['Close'] >= previous_row['Open'])

# Tạo cột 'Buy_Signal' và 'Sell_Signal' với giá trị mặc định là False, co the khong can
data['Buy_Signal'] = False
data['Sell_Signal'] = False 
    
for i in range(0, len(data)): # Chi lay 2 nen
    current_row = data.iloc[i]
    previous_row = data.iloc[i - 1]
    
    # Kiểm tra nến hiện tại có phải là nến tăng và nếu nến trước đó là nến Doji chân dài
    if is_bullish_candle(current_row, previous_row) and is_long_legged_doji(previous_row):
        # Nếu thỏa mãn cả ba điều kiện, thêm ngày vào danh sách tín hiệu mua
        data.at[current_row.name, 'Buy_Signal'] = True

    if is_bearish_candle(current_row, previous_row) and is_long_legged_doji(previous_row):
        data.at[current_row.name, 'Sell_Signal'] = True

data

,Adj Close,Close,High,Low,Open,Volume,Buy_Signal,Sell_Signal
Datetime,,,,,,,,
2025-03-17,67300.0,67300.0,67300.0,66000.0,66000.0,5003800,False,False
2025-03-18,66800.0,66800.0,67800.0,66800.0,67800.0,3757000,False,False


# Buoc 3: Show nen len de xem

In [4]:
import plotly.graph_objects as go

# Assuming 'data' is your DataFrame that includes the OHLC data and the Buy/Sell signals
# You will have to replace this with your actual data loading process

# Creating the Candlestick chart for the OHLC data
fig = go.Figure(data=[go.Candlestick(x=data.index,
                                     open=data['Open'],
                                     high=data['High'],
                                     low=data['Low'],
                                     close=data['Close'],
                                     name='OHLC')])

# Adding the Buy signals to the chart
fig.add_trace(go.Scatter(mode='markers',
                         x=data[data['Buy_Signal']].index,
                         y=data[data['Buy_Signal']]['High'],
                         marker=dict(color='green', size=10, symbol='triangle-up'),
                         name='Buy Signal'))

# Adding the Sell signals to the chart
fig.add_trace(go.Scatter(mode='markers',
                         x=data[data['Sell_Signal']].index,
                         y=data[data['Sell_Signal']]['Low'],
                         marker=dict(color='red', size=10, symbol='triangle-down'),
                         name='Sell Signal'))

# Update the layout for a better view
fig.update_layout(title='Stock Price with Buy & Sell Signals',
                  xaxis_title='Date',
                  yaxis_title='Price',
                  xaxis_rangeslider_visible=False)

# Show the figure in your Python environment
fig.show()
